# DIV 파이프라인 · Colab GPU 테스트

Colab에서 **런타임 → 런타임 유형 변경 → GPU**를 선택하고 위에서부터 실행합니다.
SSH·Docker·PostgreSQL 없이 노트북에서 기존 Supervisor → Idea → Plan/Research 그래프를 직접 호출합니다.
DB 영속화 테스트가 아니라 **대화 → 질문 응답 → 기획서 생성** 테스트입니다.

**사용자가 직접 실행:** 저장소 다운로드, Python 의존성 설치, Ollama 설치, 모델 다운로드 셀.
이 노트북은 작성 시 실제 Colab에서 실행하지 않았습니다. 모든 출력은 비워 두었습니다.
외부 설치 셀을 포함하므로 `모두 실행`보다 각 셀을 확인하며 순서대로 실행하세요.

다운로드 위치: 소스 `/content/div-project`, Python 패키지는 현재 Colab Python 환경,
Ollama는 시스템 설치 경로(`/usr` 또는 `/usr/local`), 모델 `/content/ollama-models`,
임베딩 `/content/div-project/.cache/models`. `qwen3.5:4b`는 약 3.3GB이며
Ollama 바이너리·Python 패키지·임베딩도 별도 공간이 필요합니다(버전에 따라 크기가 달라짐).
런타임 종료 시 로컬 파일이 사라질 수 있으므로 마지막 결과 다운로드 셀을 실행하세요.
공식 설치 안내: https://docs.ollama.com/linux


In [1]:
import sys, os, subprocess, json, time
from pathlib import Path
assert sys.version_info >= (3, 12), "프로젝트는 Python 3.12 이상이 필요합니다. Colab 런타임을 확인하세요."


In [2]:

subprocess.run(["nvidia-smi"], check=True)
print("Python:", sys.version)

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]


## 1. 소스 준비 — 사용자 실행
기본값은 저장 기능까지 푸시된 커밋입니다. 브랜치 최신 상태를 쓰려면 `REVISION`을 수정하세요.
비공개 저장소라면 토큰을 셀에 쓰지 말고, 접근 가능한 소스를 직접 `/content/div-project`에 준비하세요.
기존 폴더는 덮어쓰지 않습니다. 새 커밋으로 바꾸려면 별도 폴더를 사용하세요.

In [ ]:
REPO_URL = "https://github.com/vego00/pro-nlp-finalproject-nlp-04.git"
REVISION = "05c4547"
ROOT = Path("/content/div-project")
if not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
    subprocess.run(["git", "checkout", REVISION], cwd=ROOT, check=True)
assert (ROOT / "src/graph/supervisor_graph.py").is_file(), "프로젝트 소스를 확인하세요."
os.chdir(ROOT)
subprocess.run(["git", "rev-parse", "HEAD"], check=True)


## 2. Python 의존성 설치 — 사용자 실행
프로젝트에 선언된 런타임 의존성을 현재 노트북 Python에 설치합니다.
설치 후 이미 import한 패키지의 재시작 경고가 뜨면 세션을 재시작하고 1번까지 다시 실행한 뒤
설치 셀을 건너뛰세요. 설치 완료 검사는 `pip check`입니다.

In [ ]:
import tomllib
project = tomllib.loads((ROOT / "pyproject.toml").read_text())
subprocess.run([sys.executable, "-m", "pip", "install", *project["project"]["dependencies"]], check=True)
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)


## 3. Ollama 설치 — 사용자 실행
Colab GPU 런타임에 Ollama 서버가 필요합니다. 이미 설치된 경우 설치를 건너뜁니다.
공식 설치 스크립트를 파일로 받아 실행합니다. Colab에 systemd가 없어도 다음 셀에서 직접 서버를 시작합니다.

In [ ]:
import shutil
if shutil.which("ollama") is None:
    subprocess.run(["curl", "-fsSL", "https://ollama.com/install.sh", "-o", "/content/install-ollama.sh"], check=True)
    subprocess.run(["sh", "/content/install-ollama.sh"], check=True)
print("Ollama binary:", shutil.which("ollama"))


## 4. 모델 설정과 서버 시작
프로젝트 모듈을 import하기 **전에** 환경변수를 설정합니다.
설정을 바꾼 뒤에는 Python 세션을 재시작해야 모듈에 반영됩니다.
Ollama는 노트북 내부의 루프백에서만 실행하며 외부 웹 서버로 공개하지 않습니다.

In [ ]:
# MODEL Options: "qwen3.5:4b" (default, T4 16GB) or "qwen3.8:27b" (Qwen/Qwen3.8-27B, L4/A100 24GB+ GPU recommended)
MODEL = "qwen3.5:4b"
os.environ.update({
    "LLM_PROVIDER": "ollama", "LLM_MODEL": MODEL, "LLM_MINI_MODEL": MODEL,
    "OLLAMA_BASE_URL": "http://127.0.0.1:11434", "OLLAMA_HOST": "127.0.0.1:11434",
    "OLLAMA_MODELS": "/content/ollama-models", "OLLAMA_NUM_PARALLEL": "1",
    "OLLAMA_MAX_LOADED_MODELS": "1", "OLLAMA_NUM_CTX": "8192",
    "OLLAMA_THINKING": "false", "LLM_MAX_TOKENS": "4096", "LLM_TIMEOUT": "300",
    "SEARCH_PROVIDER": "duckduckgo", "EMBEDDING_MODEL": "intfloat/multilingual-e5-small",
})
import httpx
base_url = os.environ["OLLAMA_BASE_URL"]
def server_ready():
    try:
        return httpx.get(base_url + "/api/version", timeout=2).is_success
    except httpx.HTTPError:
        return False
if not server_ready():
    ollama_log = open("/content/ollama.log", "a")
    ollama_process = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=subprocess.STDOUT)
    for _ in range(60):
        if server_ready():
            break
        if ollama_process.poll() is not None:
            raise RuntimeError(Path("/content/ollama.log").read_text()[-3000:])
        time.sleep(1)
assert server_ready(), "서버 시작 실패: /content/ollama.log 확인"
print(httpx.get(base_url + "/api/version").json())


## 5. 모델·임베딩 다운로드 — 사용자 실행
Ollama 모델과 검색용 CPU 임베딩을 준비합니다. 임베딩은 처음 생성할 때 자동 다운로드됩니다.
이 셀은 외부 아티팩트를 내려받으므로 사용자가 직접 실행해야 합니다.
GPU는 Ollama 추론에 사용하고 임베딩은 프로젝트 기본값대로 CPU를 사용합니다.

In [ ]:
subprocess.run(["ollama", "pull", MODEL], check=True)
sys.path.insert(0, str(ROOT / "src")) if str(ROOT / "src") not in sys.path else None
from models.embeddings import embed_query
vector = embed_query("설치 확인")
print("임베딩 차원:", len(vector))
subprocess.run(["ollama", "list"], check=True)


## 6. 실제 응답과 GPU 사용 확인
응답이 성공한 뒤 `ollama ps`의 PROCESSOR 및 `size_vram`을 확인합니다.
CPU로 표시되면 생성 테스트를 계속하기 전에 GPU 런타임과 `/content/ollama.log`를 확인하세요.

In [ ]:
from models.llm import get_llm
response = get_llm(temperature=0, max_tokens=32).invoke("Reply only with OK.")
print(response.content)
assert response.content, "모델 응답이 비어 있습니다."
subprocess.run(["ollama", "ps"], check=True)
loaded = httpx.get(base_url + "/api/ps").json().get("models", [])
print(json.dumps(loaded, ensure_ascii=False, indent=2))
assert any(m.get("size_vram", 0) > 0 for m in loaded), "GPU 메모리 사용이 확인되지 않았습니다."


## 7. 파이프라인 초기화
이 셀을 다시 실행하면 대화 상태를 새로 시작합니다. 그래프 호출 중에는 다른 셀에서
같은 그래프를 동시에 실행하지 마세요. 검색은 외부 네트워크를 사용하며 검색 결과에 따라 달라집니다.
웹 모듈에서는 초기 상태 함수만 가져옵니다. FastAPI 서버·DB 연결은 시작하지 않습니다.

In [ ]:
from copy import deepcopy
from IPython.display import display, Markdown
from web.app import initial_state, document
from graph.supervisor_graph import supervisor_app
from config import config as settings

state = initial_state()

def run_turn(response):
    global state
    candidate = deepcopy(state)
    candidate["user_response"] = response
    for round_number in range(settings.GRAPH_MAX_ROUNDS):
        # 성공한 그래프 호출 경계만 보존합니다. 실패 시 직전 성공 상태는 남습니다.
        candidate = supervisor_app.invoke(deepcopy(candidate), {"recursion_limit": settings.GRAPH_RECURSION_LIMIT})
        state = deepcopy(candidate)
        decision = state.get("supervision", {}).get("last_decision")
        print(f"라운드 {round_number + 1}: {decision}")
        if decision != "Refresh":
            break
    else:
        raise RuntimeError("자동 진행 한도 도달. 현재 상태를 확인하세요.")
    for message in state.get("messages", [])[-4:]:
        print(f"[{message.type}] {message.content}")
    if state.get("input_request"):
        print("질문:", json.dumps(state["input_request"], ensure_ascii=False, indent=2))
    markdown = document(state)
    if markdown:
        display(Markdown(markdown))
    return state.get("supervision", {}).get("request_type")


## 8. 첫 아이디어 입력
처음에는 추가 질문이나 확인 요청이 나올 수 있습니다. 이는 정상이며 다음 셀에서 답변합니다.

In [ ]:
IDEA = "대학생이 시험 일정을 입력하면 2주 학습 계획을 만드는 웹앱을 기획해줘. 1인 개발 MVP이며 핵심 기능은 일정 입력, 할 일 체크, 진도 확인이야."
run_turn(IDEA)


## 9. 질문 응답 또는 후속 요청 — 필요할 때 반복 실행
질문 양식이 있으면 각 질문의 선택지와 입력 가이드를 보여줍니다.
직접 입력한 내용을 기존 API와 같은 응답 구조로 전달합니다. 일반 대화 상태에서는
추가 요구사항·기획서 작성 요청·수정 요청을 입력하세요. 최종 결과가 나올 때까지 반복할 수 있습니다.

In [ ]:
questions = state.get("input_request")
if state.get("supervision", {}).get("request_type") == "idea_form":
    assert isinstance(questions, list) and questions, "질문 상태를 확인하세요."
    answers = {}
    for question in questions:
        print(json.dumps(question, ensure_ascii=False, indent=2))
        answer = input(question.get("question", "답변") + " > ").strip()
        if not answer:
            raise ValueError("답변을 입력한 뒤 셀을 다시 실행하세요.")
        answers[str(question["current_section"])] = {
            "question": question.get("question", ""),
            "selected_option": "사용자 입력", "value": answer,
        }
    run_turn(answers)
else:
    followup = input("후속 요청 (예: 이 방향으로 기획서를 작성해줘) > ").strip()
    if followup:
        run_turn(followup)


## 10. 결과 저장과 다운로드
Markdown 본문을 내려받습니다. 차트 등 부가 파일은 `output/`에 별도로 남으며
이 다운로드에 포함되지 않습니다. 빈 결과면 9번 셀로 대화를 이어가세요.

In [ ]:
markdown = document(state)
assert markdown.strip(), "아직 기획서 본문이 없습니다. 질문 응답과 후속 요청을 진행하세요."
output = ROOT / "output" / "colab-test-plan.md"
output.parent.mkdir(exist_ok=True, parents=True)
output.write_text(markdown, encoding="utf-8")
print("저장:", output)
from google.colab import files
files.download(str(output))


## 선택: 이 노트북에서 시작한 Ollama 종료
종료 후 다시 실행하려면 4번 셀부터 실행하세요. 모델 파일은 런타임이 유지되는 동안 남습니다.

In [ ]:
if "ollama_process" in globals() and ollama_process.poll() is None:
    ollama_process.terminate()
    ollama_process.wait(timeout=15)
    ollama_log.close()
    print("노트북에서 시작한 Ollama를 종료했습니다.")
